# 17 — smolagents: code as action

**What you'll learn**

- That smolagents ships two agents with two *action spaces*: `ToolCallingAgent` emits JSON tool calls (your ch02 loop, packaged) and `CodeAgent` writes **Python** that calls the tools — a genuinely different execution model
- How to wire the course model into smolagents with `LiteLLMModel(model_id=MODEL)` and expose the Larkspur tools with the `@tool` decorator — the same `to_openai_tools` job you did by hand in ch02
- Why code-as-action can win: one code block composes a lookup, arithmetic, and a branch where JSON tool-calling needs many round-trips — and needs no `calc` tool, because the executor *is* the calculator
- That `CodeAgent` runs model-written code in a `LocalPythonExecutor` — the ch09 security lesson back with teeth — and what the docs' sandbox options (`e2b`, `docker`, `modal`) are for
- A machinery map from every smolagents feature back to the piece you already built — and the one row, code-as-action, that you did not

*Time: ~2 min. Cost: ~$0.02. smolagents calls run through LiteLLM, so the disk cache makes reruns ~free.*

> **Before running this notebook:** `pip install -e ".[smolagents]"` (once). It pulls in `smolagents` and its `LiteLLMModel` bridge; the model still reaches OpenRouter through the same LiteLLM you have used since ch01. Everything else stays the same.

## Two action spaces: JSON tool calls, or Python

Every agent in this course so far has acted the same way. Chapter 02's `run_agent` calls the model, the model answers with a **tool call** — a JSON object naming a tool and its arguments — the loop runs that tool, feeds the result back, and repeats. The action space is JSON. Frameworks package that loop and rename its parts, but the shape is fixed: one JSON call, one round-trip, one observation.

[smolagents](https://huggingface.co/docs/smolagents/index) offers that shape too — its `ToolCallingAgent` "writes tool calls as structured JSON," the format you already know. But its *first-class* agent is different: a `CodeAgent` "writes its actions in code (as opposed to 'agents being used to write code') to invoke tools or perform computations." The model does not emit a JSON call; it emits a **Python snippet**, and smolagents runs it. Tools are just functions in that snippet's namespace, and the [guided tour](https://huggingface.co/docs/smolagents/guided_tour) is blunt about the consequences: the code "is executed either locally (potentially unsecure) or in a secure sandbox."

That is not a cosmetic difference. The idea has a name and a paper — [CodeAct](https://arxiv.org/abs/2402.01030), "Executable Code Actions Elicit Better LLM Agents" — which argues that letting an agent act in code beats JSON tool-calling, because code composes: a snippet can loop, branch, and chain calls that JSON would need many turns to express. This chapter runs both agents on the same Larkspur ticket, watches the difference in real output, and then pays the bill code-as-action comes with.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

A smolagents run is a sequence of steps, each one a model call plus a tool call or a code execution. Because `LiteLLMModel` calls `litellm.completion` under the hood, Phoenix's LiteLLM auto-instrumentation traces every step as a span — you can watch the `CodeAgent` write, run, and observe each Python block. Two things follow from that seam. The disk cache from the config cell still applies, so reruns stay ~free; but these calls skip `shoplab.llm.complete`, so the cost `LEDGER` and tracing you built in ch03 do not see them. Optional as always — skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The model: `LiteLLMModel`, prefix and all

smolagents talks to models through a small `Model` class. `LiteLLMModel` is the one that routes through LiteLLM — the same library the whole course uses — so it takes the course's `MODEL` string *unchanged*, `openrouter/` prefix included. That is the opposite of chapter 16, where LangChain's raw `ChatOpenAI` forced us to strip the prefix: there the client spoke the OpenAI API directly, here it speaks LiteLLM. `temperature` rides through `**kwargs` down into `litellm.completion`.

In [ ]:
from smolagents import LiteLLMModel
from shoplab import world

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}

# KEEP the "openrouter/" prefix -- LiteLLMModel routes through litellm, unlike ch16's ChatOpenAI.
model = LiteLLMModel(model_id=MODEL, temperature=TEMPERATURE)
print("model:", type(model).__module__ + "." + type(model).__name__)
print("model_id:", model.model_id)

## The tools you already have, as smolagents tools

The agents need the same Larkspur lookups the ops desk always uses. smolagents' `@tool` decorator turns an annotated function into a tool: the type hints become the schema and the docstring becomes the description — with one wrinkle, an explicit `Args:` section it parses per-argument. This is exactly the job `shoplab.tools.to_openai_tools` did by hand in ch02, where you built the JSON schema from a function's signature. We expose the three read-only lookups; the risky `issue_refund` stays out of reach on purpose (a `CodeAgent` with `issue_refund` in scope could *write code that spends money* — hold that thought for the sandbox section).

In [ ]:
from smolagents import tool

@tool
def get_order(order_id: str) -> dict:
    """Look up a Larkspur order by id (items, totals, status, dates).

    Args:
        order_id: the order id, e.g. ORD-7312.
    """
    return orders.get(order_id, {"error": f"no such order {order_id}"})

@tool
def get_customer(customer_id: str) -> dict:
    """Look up a Larkspur customer by id (tier, flags, history).

    Args:
        customer_id: the customer id, e.g. CUST-07.
    """
    return customers.get(customer_id, {"error": f"no such customer {customer_id}"})

@tool
def find_policy(query: str, k: int = 2) -> list:
    """Keyword-search the 12 Larkspur store policy documents.

    Args:
        query: words describing the situation.
        k: how many policy docs to return.
    """
    return world.search_policy(query, k=k)

read_tools = [get_order, get_customer, find_policy]
print("smolagents tools:", [t.name for t in read_tools])

## The familiar shape: `ToolCallingAgent`

Start with the agent that acts the way ch02 does. `ToolCallingAgent` runs a JSON tool-calling loop: each step the model emits one tool call, smolagents runs it, and the result comes back as the next observation. We give it the fixed appendix ticket `TKT-2205` — an opened, in-window return from a member (not vip), gold decision `partial_refund` under `pol-restocking` for $170.99 — and read what it emitted off `agent.memory.steps`, the trajectory it kept (your message list, under a new name).

We bound it to a few steps and mute smolagents' console so the notebook stays readable; then we print the actual tool calls it made.

In [ ]:
import io, contextlib, re
from smolagents import ToolCallingAgent
from smolagents.monitoring import LogLevel        # LogLevel.OFF mutes the run's console

TASK = ("Triage Larkspur ticket TKT-2205. Facts: order ORD-7312, customer CUST-07, sku LK-1016, "
        "qty 1, item_condition=opened, days_since_delivery=18, requested_action=refund, "
        "evidence_photo=false. The member opened the Torrent boots, wore them one evening indoors, "
        "they pinch at the toes, repacked with tags, wants a refund to the original payment method. "
        "Look up the order and the customer, search policy for the governing rule, then give the "
        "decision, the policy id, and the refund dollar amount.")

tc_agent = ToolCallingAgent(tools=read_tools, model=model, max_steps=6, verbosity_level=LogLevel.OFF)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tc_answer = tc_agent.run(TASK)

for st in tc_agent.memory.steps:
    for c in (getattr(st, "tool_calls", None) or []):
        print(f"  JSON tool call: {c.name}({c.arguments})")
actions = [st for st in tc_agent.memory.steps if getattr(st, "step_number", None) is not None]
called = sum(1 for st in actions if getattr(st, "tool_calls", None))
print(f"{len(actions)} action steps: {called} emitted a JSON call, {len(actions) - called} produced only prose")
narrated = re.findall(r"\$?\s?\d[\d,]*\.\d{2}", str(tc_answer))
print("final (prose): narrates", narrated[-1].strip() if narrated else "no computed figure",
      "-- e.g.", " ".join(str(tc_answer).split())[:90])

> **What you should see:** several JSON tool calls — `get_order(...)`, `get_customer(...)`, `find_policy(...)` — each its own model round-trip, exactly the ch02 shape: propose a call, run it, repeat. Then it stops on a *prose* answer, because a JSON tool-caller can only name tools and read their results — it has no way to *compute* a refund, so the dollar figure it lands on (printed as `final (prose): ...`) is one it *narrated*, not calculated — eyeballed arithmetic that can land a hair off the gold (a figure like `$171.00` for what should be `$170.99`) — and on this model it often burns extra turns getting there. That is the shape to hold against the next cell, where the *same* ticket is *computed* in Python.

## The new shape: `CodeAgent` writes Python

Now the same ticket, the same model, the same tools — but a `CodeAgent`. Its action is a Python snippet, and the tools are functions it may call inside that snippet. smolagents runs the snippet in a `LocalPythonExecutor` (more on that soon), captures the result, and loops. The decisive move for teaching: we read the `code_action` off each step, so the model's actual Python is *shown*, not described. The difference between JSON and code is right there in the output.

In [ ]:
from smolagents import CodeAgent

code_agent = CodeAgent(tools=read_tools, model=model, verbosity_level=LogLevel.OFF)
print("executor:", code_agent.executor_type, "->", type(code_agent.python_executor).__name__)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    code_answer = code_agent.run(TASK)

for st in code_agent.memory.steps:
    action = getattr(st, "code_action", None)
    if action:
        print(f"--- step {st.step_number}: PYTHON action ---")
        print("   " + action.strip().replace("\n", "\n   ")[:200])
print("final (structured):", code_answer)

> **What you should see:** the code agent reads its own Python back to you — `order = get_order(order_id="ORD-7312")`, then a block that pulls the line price and applies the fee — and computes `pol-restocking` and **$170.99**, the gold amount for TKT-2205 (a 10% restocking fee on an opened, non-vip return: $189.99 x 0.90). It may label the decision `partial_refund` or `approve`-with-a-fee; what is invariant is the policy and the dollar figure it *calculated in Python*. Same ticket, same model, same tools as the cell above — the only change is that the action is *code*, and it converged to a structured answer instead of trailing off into prose. That is the CodeAct finding, live (results vary by model; the point is the mechanism, not a benchmark).

## Why code-as-action can win: composition

Look at what a code-action can do that a JSON call cannot. A JSON tool call carries exactly one function and its arguments; to look up an order, extract a price, and compute a fee, the JSON agent needs a round-trip per step — and it cannot do arithmetic at all without a tool, which is precisely why you handed it a `calc` tool in ch02 (models fumble mental math). A code-action has no such limit: the smolagents docs describe code as enabling "natural composability (function nesting, loops, conditionals)."

Give a `CodeAgent` a calculation and *only* the `get_order` tool — no `calc` — and watch it compose the whole thing in one block.

In [ ]:
calc_task = ("For Larkspur order ORD-7312, line LK-1016: compute the full item value (qty * unit "
             "price) and the refund after a 10% restocking fee, rounded to cents. Return a dict "
             "{'full': ..., 'after_fee': ...} -- in one code block if you can.")

solo = CodeAgent(tools=[get_order], model=model, verbosity_level=LogLevel.OFF)   # ONE tool, no calc tool
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    calc_answer = solo.run(calc_task)

blocks = [getattr(s, "code_action", None) for s in solo.memory.steps if getattr(s, "code_action", None)]
print("code-action blocks written:", len(blocks))
print("--- the composing block ---")
print(blocks[-1].strip())
print("result:", calc_answer)

> **What you should see:** one code-action doing the entire job — call `get_order`, loop to the `LK-1016` line, multiply `qty * unit_price_usd`, apply `* 0.9`, `round(..., 2)` — returning `{'full': 189.99, 'after_fee': 170.99}`. No `calc` tool anywhere: the executor *is* the calculator, so the arithmetic tool you built in ch02 dissolves. Often it takes two blocks — the first guesses a field name that does not exist (say `line_id`), hits a Python error, and the next block fixes the names: the error was its observation, the ch02 tool-result loop with a traceback in the tool's place. That composability is the whole case for code-as-action.

## The bill: the executor runs model-written code

Everything above rests on one fact worth staring at: a `CodeAgent` executes code the *model* wrote. Its default `executor_type` is `local`, a `LocalPythonExecutor`, and — as the run above showed — that executor runs **in this very kernel, in your process**. The guided tour says the quiet part out loud: "the LLM can generate arbitrary code that will then be executed: do not add any unsafe imports."

This is chapter 09 returning with teeth. There, an [indirect prompt injection](https://arxiv.org/abs/2302.12173) hidden in a review or an email tried to trick the model into calling a risky *tool*, and OWASP's [LLM01](https://genai.owasp.org/llmrisk/llm01-prompt-injection/) named prompt injection the top LLM risk. With a `CodeAgent`, a successful injection no longer has to smuggle a JSON call past a gate — it can write Python. The `LocalPythonExecutor` does put up guardrails: it runs a restricted interpreter that blocks imports and dangerous builtins unless you authorize them. Let us see exactly where the wall is, and how thin it can get.

In [ ]:
from smolagents.local_python_executor import LocalPythonExecutor

def try_run(executor, snippet):
    try:
        out = executor(snippet)
        return "ran -> " + str(getattr(out, "output", out))
    except Exception as e:
        return str(e).split("due to:")[-1].strip()[:90]

safe = LocalPythonExecutor(additional_authorized_imports=[])      # the default guardrail
print("arithmetic:  ", try_run(safe, "189.99 * 0.9"))
print("import os:    ", try_run(safe, "import os\nos.getcwd()"))
print("open a file: ", try_run(safe, "open('/etc/passwd').read()"))

hole = LocalPythonExecutor(additional_authorized_imports=["os"])  # one authorized import = a hole
print("authorized os:", try_run(hole, "import os\nos.name"))

> **What you should see:** the default executor runs the arithmetic but refuses `import os` (`Import of os is not allowed`) and `open(...)` (`Forbidden function evaluation`) — a real guardrail. But it is a guardrail, not a jail: authorize a single import and `os.name` returns `'posix'`, and every snippet here ran *inside this kernel*, on your filesystem, with your credentials in the environment. That is why the docs offer sandboxed executors — `executor_type="e2b"`, `"docker"`, `"modal"` — that move execution off your machine. The rule to carry out of this chapter: never run untrusted `CodeAgent` output in your own process. Code-as-action buys composability; it sells you an arbitrary-code-execution surface, and ch09's attackers know how to reach it.

## Machinery map: smolagents feature to the part you built

Line them up and smolagents stops being magic — with one honest exception. Four of these rows are packaging of machinery you built in Parts 1-3; the fifth, code-as-action, is the one execution model you did *not* build by hand, the new idea this chapter exists to teach.

| smolagents feature | Your hand-built equivalent | Built in |
|---|---|---|
| `LiteLLMModel(model_id=MODEL)` | `shoplab.llm.complete` over LiteLLM — the same provider string, prefix and all | ch01 |
| `@tool` decorator | `to_openai_tools`: a function's signature + docstring turned into a tool schema | ch02 |
| `ToolCallingAgent` (JSON call per step) | `run_agent`'s loop: model emits a tool call, `run_tool` executes it, repeat | ch02 |
| `agent.memory.steps` | the message list / trajectory you threaded through the loop by hand | ch02 |
| an executor error becoming the next observation | `run_tool` returning `{"error": ...}` as an observation the model reads | ch02 |
| arithmetic inside a code-action | the `calc` tool you gave the JSON agent because models fumble math | ch02 |
| **`CodeAgent` (code-as-action)** | **nothing — a genuinely new execution model ([CodeAct](https://arxiv.org/abs/2402.01030))** | **— (new)** |
| `LocalPythonExecutor` guardrails / remote sandboxes | the ch09 lesson: never let untrusted input drive a risky action | ch09 |

## When code-as-action fits

Read the map and the trade is clear. Code-as-action shines when the work *is* composition: multi-step calculations, chaining several tools whose outputs feed each other, looping over a list, branching on an intermediate result. On those tasks the CodeAct paper's claim shows up as fewer steps and surer answers, exactly as the two runs above did — and you get arithmetic, data wrangling, and control flow for free, no tool per operation.

It fits *badly* where the actions are a small fixed set of high-stakes, side-effecting calls — issue a refund, create a replacement, send an email. There the JSON agent's rigidity is a feature: a tool call is a named, validated, gate-able event, and the guided tour notes JSON arguments are "strictly validated, no risk of arbitrary code running." A `CodeAgent` pointed at risky tools, running in a local executor, is the ch08 approval gate and the ch09 injection defense both working against a much wider attack surface. The honest deployment answer is usually: use code-as-action for the reasoning and computation, keep the money-moving actions behind validated tools and a sandbox — and never run model-written code in the process that holds your secrets.

## Recap

| Concept | One-liner |
|---|---|
| Two action spaces | smolagents' `ToolCallingAgent` emits JSON tool calls (your ch02 loop); its `CodeAgent` writes Python — a different execution model. |
| `LiteLLMModel` | routes through LiteLLM, so it takes the course `MODEL` string with its `openrouter/` prefix intact — unlike ch16's `ChatOpenAI`. |
| `@tool` | signature + docstring to tool schema — the `to_openai_tools` job from ch02, decorated. |
| Code-as-action | the model's action is a Python snippet; tools are functions in its namespace ([CodeAct](https://arxiv.org/abs/2402.01030)). |
| Composability | one block loops, branches, and does arithmetic — no `calc` tool, where JSON needs many round-trips. |
| The executor is the risk | `LocalPythonExecutor` runs model-written code in your process; guardrails block imports, but a real sandbox (`e2b`/`docker`/`modal`) is the fix. |
| ch09, amplified | code-as-action turns a prompt injection from "smuggle a tool call" into "write arbitrary code." |

## Exercises

1. **Read the calculator's code.** Give a `CodeAgent` a calc-heavy ticket — say, a partial refund where shipping is refunded on part of a multi-item order, or a tiered discount — with only the read tools and *no* `calc` tool. Print every `code_action` and confirm it never reaches for a calculator: where did the arithmetic happen, and what would the ch02 JSON agent have had to do instead?
2. **Count the round-trips.** Run `ToolCallingAgent` and `CodeAgent` on the same ticket and compare `len(agent.memory.steps)` and the number of tool calls versus code actions. Which one converged in fewer model round-trips, and does that match the CodeAct claim? (One full pair of runs; a cent or two, free on a cached rerun.)
3. **Name the risk, then probe it.** State in one sentence which ch09 risk code-as-action amplifies and why. Then make it concrete: pass `additional_authorized_imports=["subprocess"]` (or `"*"`) to a `LocalPythonExecutor` and run a benign snippet through it to see the wall come down — or flip a `CodeAgent` to `executor_type="docker"` and note what changes. What is the smallest change that would make running a `CodeAgent` on untrusted ticket text safe?

**Next up:** chapter 18 — the team archetype. CrewAI packages your ch07 multi-agent lead-and-specialists into `Agent`s, `Task`s, and a `Crew`, and because its dependency pins conflict with the main stack, it runs in its own venv with a registered kernel — the same triage, delegated across roles.